# 3D U-Net Prediction

Load a trained 3D U-Net from `scripts/unet.py`, predict on one denoised nuclei volume, and inspect the result in napari.

In [1]:
import sys
from pathlib import Path

import napari
import numpy as np
import torch
from tifffile import imread, imwrite

sys.path.append(str(Path('../scripts').resolve()))
from unet import UNet3D, load_config

In [6]:
CONFIG_PATH = Path('../configs/unet.json')
IMAGE_PATH = Path('../DATA/260330_ID479_laminB3/ST24/ST24_emb1_1_2026_03_30__10_28_07_n2v_3d.tif')
THRESHOLD = 0.5
SAVE_PRED = False

config = load_config(str(CONFIG_PATH))
MODEL_PATH = Path('../') / config['model_dir'] / f"{config['model_name']}_best.pth"
OUT_PATH = IMAGE_PATH.with_name(IMAGE_PATH.stem + '_unet_pred.tif')

In [3]:
def normalize_volume(img):
    img = np.asarray(img, dtype=np.float32)
    return (img - img.min()) / (img.max() - img.min() + 1e-8)


def predict_sliding_window(model, volume, patch_size, device):
    pz, py, px = patch_size
    z_step = max(1, pz // 2)
    y_step = max(1, py // 2)
    x_step = max(1, px // 2)

    z_max, y_max, x_max = volume.shape
    prob_sum = np.zeros((z_max, y_max, x_max), dtype=np.float32)
    count_map = np.zeros((z_max, y_max, x_max), dtype=np.float32)

    model.eval()
    with torch.no_grad():
        for z0 in range(0, max(1, z_max - pz + 1), z_step):
            z0 = min(z0, max(0, z_max - pz))
            for y0 in range(0, max(1, y_max - py + 1), y_step):
                y0 = min(y0, max(0, y_max - py))
                for x0 in range(0, max(1, x_max - px + 1), x_step):
                    x0 = min(x0, max(0, x_max - px))
                    patch = volume[z0:z0+pz, y0:y0+py, x0:x0+px]

                    if patch.shape != (pz, py, px):
                        pad = np.zeros((pz, py, px), dtype=np.float32)
                        pad[:patch.shape[0], :patch.shape[1], :patch.shape[2]] = patch
                        patch = pad

                    tensor = torch.from_numpy(patch).unsqueeze(0).unsqueeze(0).to(device)
                    pred = torch.sigmoid(model(tensor)).squeeze().cpu().numpy()

                    z1 = min(z0 + pz, z_max)
                    y1 = min(y0 + py, y_max)
                    x1 = min(x0 + px, x_max)
                    pred_crop = pred[:z1-z0, :y1-y0, :x1-x0]

                    prob_sum[z0:z1, y0:y1, x0:x1] += pred_crop
                    count_map[z0:z1, y0:y1, x0:x1] += 1

    return prob_sum / np.maximum(count_map, 1)

In [7]:
patch_size = tuple(config['patch_size'])
device = 'cuda' if torch.cuda.is_available() else 'cpu'

model = UNet3D().to(device)
state = torch.load(MODEL_PATH, map_location=device)
model.load_state_dict(state)
model.eval()

nuclei = normalize_volume(imread(IMAGE_PATH))
prob = predict_sliding_window(model, nuclei, patch_size=patch_size, device=device)
seg = (prob > THRESHOLD).astype(np.uint8)

print('device    :', device)
print('model     :', MODEL_PATH)
print('image     :', IMAGE_PATH)
print('shape     :', nuclei.shape)
print('patch_size:', patch_size)
print('fg voxels :', int(seg.sum()))

device    : cuda
model     : ..\models\unet3d_nuclei_best.pth
image     : ..\DATA\260330_ID479_laminB3\ST24\ST24_emb1_1_2026_03_30__10_28_07_n2v_3d.tif
shape     : (111, 1200, 1200)
patch_size: (32, 256, 256)
fg voxels : 21286778


In [10]:
viewer = napari.Viewer()
viewer.add_image(nuclei, name='nuclei_norm', contrast_limits=(0, 1))
viewer.add_image(prob, name='unet_prob', contrast_limits=(0, 1), opacity=0.6)
viewer.add_labels(seg, name='unet_seg')

c:\Users\ljd567\AppData\Local\miniconda3\envs\torch_env\lib\site-packages\napari\plugins\_plugin_manager.py:555: UserWarning: Plugin 'napari_skimage_regionprops2' has already registered a function widget 'duplicate current frame' which has now been overwritten
  warn(message=warn_message)


<Labels layer 'unet_seg' at 0x217c423c8b0>

In [ ]:
if SAVE_PRED:
    imwrite(OUT_PATH, seg.astype(np.uint8))
    print('saved:', OUT_PATH)